In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # 🔐 03 · Clasificación y Seguridad — Unity Catalog
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Aplicar clasificación de sensibilidad a columnas críticas del dominio
# MAGIC **Trust & Safety** mediante tags de Unity Catalog.
# MAGIC
# MAGIC Este notebook:
# MAGIC
# MAGIC - **NO transforma datos**;
# MAGIC - **NO crea tablas**;
# MAGIC - **NO recalcula riesgo**;
# MAGIC - **NO modifica lógica funcional**;
# MAGIC - solo administra metadata de clasificación.
# MAGIC
# MAGIC ## Alcance
# MAGIC
# MAGIC - `workspace.yelp_silver.slv_review`
# MAGIC - `workspace.yelp_gold.gld_review_risk`
# MAGIC - `workspace.yelp_gold.gld_investigation_queue`
# MAGIC
# MAGIC ## Taxonomía utilizada
# MAGIC
# MAGIC | Clasificación | Uso |
# MAGIC |---|---|
# MAGIC | `internal` | Uso interno, sin sensibilidad elevada |
# MAGIC | `restricted` | Identificadores o datos que requieren acceso restringido |
# MAGIC | `confidential` | Texto libre, señales, scores o resultados de riesgo |
# MAGIC
# MAGIC > Los tags de clasificación **no reemplazan** permisos, masking,
# MAGIC > row filters u otras políticas de seguridad.

# COMMAND ----------
from pyspark.sql import functions as F
# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOLD_SCHEMA = "yelp_gold"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

SILVER_REVIEW = f"{SILVER}.slv_review"
GOLD_REVIEW_RISK = f"{GOLD}.gld_review_risk"
GOLD_INVESTIGATION_QUEUE = f"{GOLD}.gld_investigation_queue"

PIPELINE_NAME = "03_clasificacion_seguridad_uc"

print("=" * 78)
print("YELPCONNECT - CLASIFICACIÓN Y SEGURIDAD UNITY CATALOG")
print("=" * 78)
print(f"Pipeline : {PIPELINE_NAME}")
print(f"Catalog  : {CATALOG}")
print("=" * 78)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Precondiciones
# MAGIC
# MAGIC Las tablas deben existir antes de aplicar clasificación.
# MAGIC
# MAGIC Este notebook debe ejecutarse **después de `06_gold_productos`**
# MAGIC y **antes de `07_validacion_end_to_end`**.

# COMMAND ----------

# ============================================================
# 2. VALIDAR TABLAS REQUERIDAS
# ============================================================

required_tables = [
    SILVER_REVIEW,
    GOLD_REVIEW_RISK,
    GOLD_INVESTIGATION_QUEUE
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]

if missing_tables:

    print("❌ Tablas requeridas no encontradas:")

    for table_name in missing_tables:
        print(f" - {table_name}")

    raise Exception(
        "No se puede aplicar clasificación de seguridad."
    )

print("✔ Tablas requeridas disponibles.")


# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Matriz de clasificación
# MAGIC
# MAGIC La clasificación se centraliza en una única estructura para evitar
# MAGIC duplicar lógica entre Silver, Gold y el Acceptance Gate.

# COMMAND ----------

# ============================================================
# 3. MATRIZ DE CLASIFICACIÓN
# ============================================================

CLASSIFICATION_MATRIX = {

    SILVER_REVIEW: {

        "user_id":
            "restricted",

        "review_text":
            "confidential",

        "review_id":
            "internal",

        "business_id":
            "internal"
    },


    GOLD_REVIEW_RISK: {

        "user_id":
            "restricted",

        "review_text":
            "confidential",

        "signal_rules":
            "confidential",

        "weighted_risk_score":
            "confidential",

        "risk_classification":
            "confidential",

        "investigation_priority":
            "confidential"
    },


    GOLD_INVESTIGATION_QUEUE: {

        "user_id":
            "restricted",

        "user_sk":
            "restricted",

        "review_text":
            "confidential",

        "signal_rules":
            "confidential",

        "weighted_risk_score":
            "confidential",

        "risk_classification":
            "confidential",

        "investigation_priority":
            "confidential"
    }
}


# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Validar columnas antes de aplicar tags
# MAGIC
# MAGIC Si una columna definida en la matriz no existe, el notebook se detiene.
# MAGIC
# MAGIC Esto evita que una clasificación quede parcialmente aplicada.

# COMMAND ----------

# ============================================================
# 4. VALIDAR COLUMNAS
# ============================================================

missing_columns = []

for table_name, column_map in CLASSIFICATION_MATRIX.items():

    existing_columns = {
        field.name
        for field in spark.table(table_name).schema.fields
    }

    for column_name in column_map:

        if column_name not in existing_columns:

            missing_columns.append(
                (
                    table_name,
                    column_name
                )
            )


if missing_columns:

    print("❌ Columnas requeridas no encontradas:")

    for table_name, column_name in missing_columns:

        print(
            f" - {table_name}.{column_name}"
        )

    raise Exception(
        "No se puede aplicar clasificación: "
        "la matriz no coincide con el esquema."
    )


print("✔ Columnas requeridas disponibles.")


# COMMAND ----------

# MAGIC %md
# MAGIC # 5. Aplicar clasificación
# MAGIC
# MAGIC Se utiliza `ALTER COLUMN ... SET TAGS`.
# MAGIC
# MAGIC La operación es idempotente para esta taxonomía: volver a ejecutar el
# MAGIC notebook mantiene el valor esperado del tag `classification`.

# COMMAND ----------

# ============================================================
# 5. APLICAR TAGS DE CLASIFICACIÓN
# ============================================================

applied = []

for table_name, column_map in CLASSIFICATION_MATRIX.items():

    for column_name, classification in column_map.items():

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            ALTER COLUMN {column_name}
            SET TAGS (
                'classification' = '{classification}'
            )
            """
        )

        applied.append(
            (
                table_name,
                column_name,
                classification
            )
        )

        print(
            f"✔ {table_name}.{column_name} "
            f"→ {classification}"
        )


# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Resultado aplicado
# MAGIC
# MAGIC Se muestra la matriz efectiva de clasificación aplicada por el notebook.

# COMMAND ----------

# ============================================================
# 6. MATRIZ APLICADA
# ============================================================

applied_df = spark.createDataFrame(
    applied,
    [
        "table_name",
        "column_name",
        "classification"
    ]
)

display(
    applied_df
    .orderBy(
        "table_name",
        "column_name"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 7. Validación en Unity Catalog
# MAGIC
# MAGIC La fuente de verdad para validar los tags es:
# MAGIC
# MAGIC `workspace.information_schema.column_tags`
# MAGIC
# MAGIC El notebook verifica que cada columna tenga exactamente la
# MAGIC clasificación esperada.

# COMMAND ----------

# ============================================================
# 7. LEER TAGS DESDE INFORMATION_SCHEMA
# ============================================================

classification_tags = spark.sql(
    f"""
    SELECT
        schema_name,
        table_name,
        column_name,
        tag_name,
        tag_value
    FROM {CATALOG}.information_schema.column_tags
    WHERE schema_name IN (
        '{SILVER_SCHEMA}',
        '{GOLD_SCHEMA}'
    )
      AND tag_name = 'classification'
    """
)


display(
    classification_tags
    .orderBy(
        "schema_name",
        "table_name",
        "column_name"
    )
)


# COMMAND ----------

# ============================================================
# 8. VALIDAR MATRIZ ESPERADA VS UNITY CATALOG
# ============================================================

expected = {}


for table_name, column_map in CLASSIFICATION_MATRIX.items():

    parts = table_name.split(".")

    schema_name = parts[1]
    short_table_name = parts[2]

    for column_name, classification in column_map.items():

        expected[
            (
                schema_name,
                short_table_name,
                column_name
            )
        ] = classification


observed = {

    (
        row.schema_name,
        row.table_name,
        row.column_name
    ):
        row.tag_value

    for row in classification_tags.collect()
}


validation_rows = []

for key, expected_value in expected.items():

    actual_value = observed.get(
        key
    )

    passed = (
        actual_value == expected_value
    )

    validation_rows.append(
        (
            key[0],
            key[1],
            key[2],
            expected_value,
            actual_value,
            passed
        )
    )


validation_df = spark.createDataFrame(
    validation_rows,
    [
        "schema_name",
        "table_name",
        "column_name",
        "expected_classification",
        "actual_classification",
        "passed"
    ]
)


display(
    validation_df
    .orderBy(
        "schema_name",
        "table_name",
        "column_name"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 9. Quality Gate de clasificación
# MAGIC
# MAGIC Si alguna clasificación esperada no está presente o tiene un valor
# MAGIC distinto, el notebook termina `FAILED`.
# MAGIC
# MAGIC El Quality Gate **reconstruye su propia evidencia desde**
# MAGIC `workspace.information_schema.column_tags`, por lo que no depende de
# MAGIC variables creadas por una celda anterior.
# MAGIC
# MAGIC Esto permite ejecutar el notebook de forma segura con `Run all`
# MAGIC o como tarea dentro de Databricks Workflows.
# MAGIC
# MAGIC Esto garantiza que `07_validacion_end_to_end` reciba una capa de
# MAGIC gobierno consistente.

# COMMAND ----------

# ============================================================
# 9. QUALITY GATE AUTOSUFICIENTE
# ============================================================
#
# Este bloque reconstruye la validación desde Unity Catalog.
# No depende de que validation_df exista previamente.
# ============================================================

classification_tags_gate = spark.sql(
    f"""
    SELECT
        schema_name,
        table_name,
        column_name,
        tag_name,
        tag_value
    FROM {CATALOG}.information_schema.column_tags
    WHERE schema_name IN (
        '{SILVER_SCHEMA}',
        '{GOLD_SCHEMA}'
    )
      AND tag_name = 'classification'
    """
)


expected_gate = {}

for table_name, column_map in CLASSIFICATION_MATRIX.items():

    parts = table_name.split(".")

    schema_name = parts[1]
    short_table_name = parts[2]

    for column_name, classification in column_map.items():

        expected_gate[
            (
                schema_name,
                short_table_name,
                column_name
            )
        ] = classification


observed_gate = {

    (
        row.schema_name,
        row.table_name,
        row.column_name
    ):
        row.tag_value

    for row in classification_tags_gate.collect()
}


validation_rows_gate = []

for key, expected_value in expected_gate.items():

    actual_value = observed_gate.get(
        key
    )

    passed = (
        actual_value == expected_value
    )

    validation_rows_gate.append(
        (
            key[0],
            key[1],
            key[2],
            expected_value,
            actual_value,
            passed
        )
    )


validation_df = spark.createDataFrame(
    validation_rows_gate,
    [
        "schema_name",
        "table_name",
        "column_name",
        "expected_classification",
        "actual_classification",
        "passed"
    ]
)


display(
    validation_df
    .orderBy(
        "schema_name",
        "table_name",
        "column_name"
    )
)


failed_classifications = (
    validation_df
    .filter(
        F.col(
            "passed"
        ) == False
    )
    .count()
)


total_classifications = (
    validation_df.count()
)


passed_classifications = (
    total_classifications
    -
    failed_classifications
)


print()
print("=" * 78)
print("RESULTADO CLASIFICACIÓN UNITY CATALOG")
print("=" * 78)

print(
    f"Clasificaciones esperadas : "
    f"{total_classifications}"
)

print(
    f"Clasificaciones correctas : "
    f"{passed_classifications}"
)

print(
    f"Clasificaciones fallidas  : "
    f"{failed_classifications}"
)

print("=" * 78)


if failed_classifications > 0:

    print()
    print("❌ Clasificaciones incorrectas o faltantes:")

    display(
        validation_df
        .filter(
            F.col(
                "passed"
            ) == False
        )
        .orderBy(
            "schema_name",
            "table_name",
            "column_name"
        )
    )

    raise Exception(
        f"""
        ❌ QUALITY GATE DE CLASIFICACIÓN FALLÓ.

        Clasificaciones incorrectas:
        {failed_classifications}
        """
    )


print()
print(
    "✔ QUALITY GATE DE CLASIFICACIÓN SUPERADO"
)

print(
    "✔ Unity Catalog contiene la clasificación esperada."
)

print(
    "✔ Siguiente etapa: 07_validacion_end_to_end"
)

# COMMAND ----------

# MAGIC %md
# MAGIC # ✅ Definition of Done
# MAGIC
# MAGIC El notebook se considera completo cuando:
# MAGIC
# MAGIC - [ ] `slv_review` tiene clasificación en columnas sensibles.
# MAGIC - [ ] `gld_review_risk` tiene clasificación en identificadores y riesgo.
# MAGIC - [ ] `gld_investigation_queue` tiene clasificación en identificadores y riesgo.
# MAGIC - [ ] `user_id` está clasificado como `restricted`.
# MAGIC - [ ] `user_sk` está clasificado como `restricted`.
# MAGIC - [ ] `review_text` está clasificado como `confidential`.
# MAGIC - [ ] `weighted_risk_score` está clasificado como `confidential`.
# MAGIC - [ ] `risk_classification` está clasificado como `confidential`.
# MAGIC - [ ] `investigation_priority` está clasificado como `confidential`.
# MAGIC - [ ] La validación contra `information_schema.column_tags` no presenta diferencias.
# MAGIC - [ ] El Quality Gate de clasificación termina correctamente.
